In [3]:
import pandas as pd

df = pd.read_csv("churn-bigml-80.csv")

df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [4]:
print("Number of rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Number of rows and columns: (2666, 20)

Column names:
['State', 'Account length', 'Area code', 'International plan', 'Voice mail plan', 'Number vmail messages', 'Total day minutes', 'Total day calls', 'Total day charge', 'Total eve minutes', 'Total eve calls', 'Total eve charge', 'Total night minutes', 'Total night calls', 'Total night charge', 'Total intl minutes', 'Total intl calls', 'Total intl charge', 'Customer service calls', 'Churn']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   object 
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   object 
 4   Voice mail plan         2666 non-null   object 
 5   Number vmail messages   2666 non-null   int64  
 6

In [5]:

print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())

Missing values in each column:
State                     0
Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
dtype: int64

Total missing values: 0

Number of duplicate rows: 0


In [7]:
# Identify categorical columns
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

# Identify numerical columns
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("\nNumerical columns:")
print(numerical_columns)

Categorical columns:
['State', 'International plan', 'Voice mail plan']

Numerical columns:
['Account length', 'Area code', 'Number vmail messages', 'Total day minutes', 'Total day calls', 'Total day charge', 'Total eve minutes', 'Total eve calls', 'Total eve charge', 'Total night minutes', 'Total night calls', 'Total night charge', 'Total intl minutes', 'Total intl calls', 'Total intl charge', 'Customer service calls']


In [8]:
print("\nUnique values in categorical columns:")

for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].unique())



Unique values in categorical columns:

State:
['KS' 'OH' 'NJ' 'OK' 'AL' 'MA' 'MO' 'WV' 'RI' 'IA' 'MT' 'ID' 'VT' 'VA'
 'TX' 'FL' 'CO' 'AZ' 'NE' 'WY' 'IL' 'NH' 'LA' 'GA' 'AK' 'MD' 'AR' 'WI'
 'OR' 'DE' 'IN' 'UT' 'CA' 'SD' 'NC' 'WA' 'MN' 'NM' 'NV' 'DC' 'NY' 'KY'
 'ME' 'MS' 'MI' 'SC' 'TN' 'PA' 'HI' 'ND' 'CT']

International plan:
['No' 'Yes']

Voice mail plan:
['Yes' 'No']


In [9]:
# Check unique values of Area code
print("Unique Area Codes:")
print(df['Area code'].unique())

print("\nNumber of unique Area Codes:")
print(df['Area code'].nunique())

# Check the target variable
print("\nChurn values:")
print(df['Churn'].unique())

print("\nChurn distribution:")
print(df['Churn'].value_counts())


Unique Area Codes:
[415 408 510]

Number of unique Area Codes:
3

Churn values:
[False  True]

Churn distribution:
Churn
False    2278
True      388
Name: count, dtype: int64


In [10]:
continuous_columns = [
    'Total day minutes',
    'Total day charge',
    'Total eve minutes',
    'Total eve charge',
    'Total night minutes',
    'Total night charge',
    'Total intl minutes',
    'Total intl charge'
]

print("Continuous columns selected for outlier analysis:")
for column in continuous_columns:
    print("-", column)

Continuous columns selected for outlier analysis:
- Total day minutes
- Total day charge
- Total eve minutes
- Total eve charge
- Total night minutes
- Total night charge
- Total intl minutes
- Total intl charge


In [11]:
# Detect outliers using the IQR method

outlier_summary = []

for column in continuous_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ).sum()

    outlier_summary.append({
        'Column': column,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outlier Count': outlier_count
    })

outlier_df = pd.DataFrame(outlier_summary)

outlier_df

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Total day minutes,143.4000,215.900,72.5000,34.65000,324.65000,21
1,Total day charge,24.3800,36.700,12.3200,5.90000,55.18000,21
2,Total eve minutes,165.3000,235.100,69.8000,60.60000,339.80000,17
3,Total eve charge,14.0500,19.980,5.9300,5.15500,28.87500,17
4,Total night minutes,166.9250,236.475,69.5500,62.60000,340.80000,22
5,Total night charge,7.5125,10.640,3.1275,2.82125,15.33125,22
6,Total intl minutes,8.5000,12.100,3.6000,3.10000,17.50000,37
7,Total intl charge,2.3000,3.270,0.9700,0.84500,4.72500,40


In [12]:
print("Outlier count for each column:")

for column in continuous_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(f"{column}: {len(outliers)} outliers")

Outlier count for each column:
Total day minutes: 21 outliers
Total day charge: 21 outliers
Total eve minutes: 17 outliers
Total eve charge: 17 outliers
Total night minutes: 22 outliers
Total night charge: 22 outliers
Total intl minutes: 37 outliers
Total intl charge: 40 outliers


In [13]:
for column in continuous_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ][column]

    print(f"\n{column}")
    print("Minimum outlier:", outliers.min())
    print("Maximum outlier:", outliers.max())


Total day minutes
Minimum outlier: 0.0
Maximum outlier: 350.8

Total day charge
Minimum outlier: 0.0
Maximum outlier: 59.64

Total eve minutes
Minimum outlier: 0.0
Maximum outlier: 363.7

Total eve charge
Minimum outlier: 0.0
Maximum outlier: 30.91

Total night minutes
Minimum outlier: 43.7
Maximum outlier: 395.0

Total night charge
Minimum outlier: 1.97
Maximum outlier: 17.77

Total intl minutes
Minimum outlier: 0.0
Maximum outlier: 20.0

Total intl charge
Minimum outlier: 0.0
Maximum outlier: 5.4


In [14]:
df_clean = df.copy()

for column in continuous_columns:
    Q1 = df_clean[column].quantile(0.25)
    Q3 = df_clean[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_clean = df_clean[
        (df_clean[column] >= lower_bound) &
        (df_clean[column] <= upper_bound)
    ]

print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)
print("Rows removed:", len(df) - len(df_clean))

Original dataset shape: (2666, 20)
Cleaned dataset shape: (2564, 20)
Rows removed: 102


In [15]:

print("Missing values after cleaning:")
print(df_clean.isnull().sum().sum())

print("\nDuplicate rows after cleaning:")
print(df_clean.duplicated().sum())

print("\nCleaned dataset shape:")
print(df_clean.shape)

Missing values after cleaning:
0

Duplicate rows after cleaning:
0

Cleaned dataset shape:
(2564, 20)


In [17]:
# Make another copy before encoding
df_encoded = df_clean.copy()

# Convert Yes/No columns to 0/1
df_encoded['International plan'] = df_encoded['International plan'].map({
    'No': 0,
    'Yes': 1
})

df_encoded['Voice mail plan'] = df_encoded['Voice mail plan'].map({
    'No': 0,
    'Yes': 1
})

# One-hot encode State
df_encoded = pd.get_dummies(
    df_encoded,
    columns=['State'],
    dtype=int
)

print("Dataset shape after encoding:", df_encoded.shape)
print("\nFirst 5 rows:")
df_encoded.head()


Dataset shape after encoding: (2564, 70)

First 5 rows:


,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,...,State_SD,State_TN,State_TX,State_UT,State_VA,State_VT,State_WA,State_WI,State_WV,State_WY
0,128,415,0,1,25,265.1,110,45.07,197.4,99,...,0,0,0,0,0,0,0,0,0,0
1,107,415,0,1,26,161.6,123,27.47,195.5,103,...,0,0,0,0,0,0,0,0,0,0
2,137,415,0,0,0,243.4,114,41.38,121.2,110,...,0,0,0,0,0,0,0,0,0,0
3,84,408,1,0,0,299.4,71,50.90,61.9,88,...,0,0,0,0,0,0,0,0,0,0
4,75,415,1,0,0,166.7,113,28.34,148.3,122,...,0,0,0,0,0,0,0,0,0,0


In [18]:
print("Data types after encoding:")
print(df_encoded.dtypes.value_counts())

print("\nRemaining categorical columns:")
print(df_encoded.select_dtypes(include=['object']).columns.tolist())

Data types after encoding:
int64      61
float64     8
bool        1
Name: count, dtype: int64

Remaining categorical columns:
[]


In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Keep the target column separate
target = df_encoded['Churn']

features = df_encoded.drop(columns=['Churn'])

# Standardize all features
scaled_features = scaler.fit_transform(features)

# Convert back to DataFrame
df_scaled = pd.DataFrame(
    scaled_features,
    columns=features.columns,
    index=features.index
)

# Add target column back
df_scaled['Churn'] = target

print("Final dataset shape:", df_scaled.shape)
df_scaled.head()

Final dataset shape: (2564, 70)


,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,...,State_TN,State_TX,State_UT,State_VA,State_VT,State_WA,State_WI,State_WV,State_WY,Churn
0,0.697160,-0.528348,-0.333044,1.615942,1.236292,1.622616,0.486256,1.622902,-0.065483,-0.051122,...,-0.127477,-0.146676,-0.152133,-0.158713,-0.148058,-0.136649,-0.149427,-0.187412,-0.158713,False
1,0.166289,-0.528348,-0.333044,1.615942,1.309448,-0.348125,1.144043,-0.348411,-0.103739,0.147857,...,-0.127477,-0.146676,-0.152133,-0.158713,-0.148058,-0.136649,-0.149427,-0.187412,-0.158713,False
2,0.924676,-0.528348,-0.333044,-0.618834,-0.592610,1.209427,0.688652,1.209598,-1.599741,0.496072,...,-0.127477,-0.146676,-0.152133,-0.158713,-0.148058,-0.136649,-0.149427,-0.187412,-0.158713,False
3,-0.415141,-0.692917,3.002603,-0.618834,-0.592610,2.275721,-1.487106,2.275899,-2.793724,-0.598317,...,-0.127477,-0.146676,-0.152133,-0.158713,-0.148058,-0.136649,-0.149427,-0.187412,-0.158713,False
4,-0.642657,-0.528348,3.002603,-0.618834,-0.592610,-0.251016,0.638053,-0.250965,-1.054093,1.093011,...,-0.127477,-0.146676,-0.152133,-0.158713,-0.148058,-0.136649,-0.149427,-0.187412,-0.158713,False


In [21]:
print("Mean of first 10 standardized features:")
print(df_scaled[features.columns[:10]].mean().round(2))

print("\nStandard deviation of first 10 standardized features:")
print(df_scaled[features.columns[:10]].std().round(2))

Mean of first 10 standardized features:
Account length          -0.0
Area code               -0.0
International plan      -0.0
Voice mail plan         -0.0
Number vmail messages   -0.0
Total day minutes       -0.0
Total day calls         -0.0
Total day charge         0.0
Total eve minutes        0.0
Total eve calls          0.0
dtype: float64

Standard deviation of first 10 standardized features:
Account length           1.0
Area code                1.0
International plan       1.0
Voice mail plan          1.0
Number vmail messages    1.0
Total day minutes        1.0
Total day calls          1.0
Total day charge         1.0
Total eve minutes        1.0
Total eve calls          1.0
dtype: float64
